# Silver → Gold | Modelo Dimensional (Esquema Estrela) com Surrogate Keys

Notebook PySpark (Fabric) que lê as tabelas da camada **Silver**, gera **surrogate keys (SK)** para cada dimensão e monta a tabela **fato** relacionando-se com as dimensões exclusivamente através das SKs (nunca pelas chaves de negócio do SAP).

Inclui também o padrão de **membro desconhecido** (unknown member): cada dimensão recebe uma linha extra com `SK = -1` e colunas descritivas preenchidas com `"N/A"`, usada como destino padrão sempre que a fato não encontrar correspondência real na dimensão.

## Modelo Dimensional — Visão Conceitual

**Tipo de modelo:** esquema estrela (star schema)
**Fato:** `fato_despesas`
**Grão da fato:** um item de lançamento contábil (ACDOCA)

```
FATO_DESPESAS   (grão: 1 item de lançamento contábil — ACDOCA)
│
├── sk_empresa            ──► dim_empresa
├── sk_conta_contabil     ──► dim_conta_contabil
├── sk_segmento           ──► dim_segmento
├── sk_centro_custo       ──► dim_centro_custo
├── sk_centro_lucro       ──► dim_centro_lucro
└── sk_cliente_fornecedor ──► dim_cliente_fornecedor   (papel: Cliente)
```

**Dimensão de papéis (role-playing dimension):** `dim_cliente_fornecedor` é a mesma tabela de Business Partner e pode representar tanto o papel de **Cliente** quanto o de **Fornecedor**. Nesta extração, a fato só carrega o campo `parceiro_negocio` (via `cliente`/Customer) — não há campo de Fornecedor (Supplier) no ACDOCA atual. Por isso a fato tem uma única FK, `sk_cliente_fornecedor`, apontando para o papel de Cliente. Se uma extração futura trouxer o campo Supplier, basta repetir o mesmo padrão de join contra a mesma dimensão, guardando o resultado em outra coluna.

**Membro desconhecido (SK = -1):** toda dimensão recebe, além dos registros reais, uma linha adicional com `SK = -1` e `"N/A"` nas colunas descritivas. Quando um lançamento da fato tem uma chave de negócio nula (ou sem correspondência na dimensão), a FK correspondente recebe `-1` em vez de ficar nula — assim toda análise/join fato↔dimensão sempre encontra uma linha válida do outro lado, mesmo para dados incompletos.

| Tabela | Tipo | Chave (PK) | Origem (silver) |
|---|---|---|---|
| `fato_despesas` | Fato | sk_* (FKs) + `id_lancamento` (chave natural, mantida como atributo degenerado) | `lancamento_despesas` |
| `dim_empresa` | Dimensão | `sk_empresa` | `empresa` |
| `dim_conta_contabil` | Dimensão | `sk_conta_contabil` | `conta_contabil` |
| `dim_segmento` | Dimensão | `sk_segmento` | `segmento` |
| `dim_centro_custo` | Dimensão | `sk_centro_custo` | `centro_custo` |
| `dim_centro_lucro` | Dimensão | `sk_centro_lucro` | `centro_lucro` |
| `dim_cliente_fornecedor` | Dimensão (papéis) | `sk_cliente_fornecedor` | `cliente_fornecedor` |

**Por que usar surrogate keys em vez da chave de negócio do SAP diretamente?**
- Desacopla o modelo analítico de mudanças/reuso de códigos no sistema de origem
- Permite representar histórico (SCD) no futuro sem alterar a estrutura da fato
- Chaves inteiras sequenciais são mais leves para join/índice do que strings compostas
- Uma mesma dimensão pode ser referenciada mais de uma vez pela fato em papéis diferentes (ex: Cliente/Fornecedor), o que não seria possível de forma limpa usando só a chave de negócio
- Permite o padrão de membro desconhecido (SK = -1), que não existiria de forma natural usando só a chave de negócio do SAP


In [1]:
# Bibliotecas do PySpark utilizadas no notebook
from pyspark.sql import SparkSession, Row
from pyspark.sql.functions import col, row_number, when, sum as spark_sum, coalesce, lit, make_date, year, month
from pyspark.sql.window import Window
from pyspark.sql.types import StringType

spark = SparkSession.builder.getOrCreate()


StatementMeta(, cf04796a-c702-461d-8ed8-9c53905e0d33, 3, Finished, Available, Finished, False)

In [2]:
# ---------------------------------------------------------------
# PARÂMETROS DO NOTEBOOK
# ---------------------------------------------------------------
LAKEHOUSE_SILVER = "lh_sap_silver"   # lakehouse de origem (dados tratados)
LAKEHOUSE_GOLD = "lh_sap_gold"       # lakehouse de destino (modelo dimensional)
SCHEMA_SILVER = "dbo"
SCHEMA_GOLD = "dbo"

SK_DESCONHECIDO = -1          # valor de SK para o membro desconhecido
VALOR_DESCONHECIDO_TEXTO = "N/A"   # valor usado nas colunas de texto do membro desconhecido


StatementMeta(, cf04796a-c702-461d-8ed8-9c53905e0d33, 4, Finished, Available, Finished, False)

## 1. Configuração das dimensões

Para cada dimensão: de qual tabela silver ela vem, qual é a chave de negócio (usada para gerar a SK e depois relacionar com a fato) e qual nome a SK e a tabela gold vão ter.

In [3]:
CONFIG_DIMENSOES = {

    # tabela_silver: {chave usada para gerar/relacionar a SK, nome da SK, nome da tabela gold}
    "empresa": {
        "chave_negocio": ["empresa"],
        "nome_sk": "sk_empresa",
        "tabela_gold": "dim_empresa",
    },
    "conta_contabil": {
        # Observação: a fato não carrega "plano_contas" (ChartOfAccounts) nesta
        # extração, então a chave de junção usa só "conta_contabil". Se a mesma
        # conta puder existir em mais de um plano de contas, inclua "plano_contas"
        # aqui E na extração da fato para evitar ambiguidade.
        "chave_negocio": ["conta_contabil"],
        "nome_sk": "sk_conta_contabil",
        "tabela_gold": "dim_conta_contabil",
    },
    "segmento": {
        "chave_negocio": ["segmento"],
        "nome_sk": "sk_segmento",
        "tabela_gold": "dim_segmento",
    },
    "centro_custo": {
        "chave_negocio": ["centro_custo"],
        "nome_sk": "sk_centro_custo",
        "tabela_gold": "dim_centro_custo",
    },
    "centro_lucro": {
        "chave_negocio": ["centro_lucro"],
        "nome_sk": "sk_centro_lucro",
        "tabela_gold": "dim_centro_lucro",
    },
    "cliente_fornecedor": {
        "chave_negocio": ["parceiro_negocio"],
        "nome_sk": "sk_cliente_fornecedor",
        "tabela_gold": "dim_cliente_fornecedor",
    },
}


StatementMeta(, cf04796a-c702-461d-8ed8-9c53905e0d33, 5, Finished, Available, Finished, False)

## 2. Funções utilitárias — SK e membro desconhecido

In [4]:
def gerar_dim_com_sk(df, chave_negocio: list, nome_sk: str):
    """
    Gera uma surrogate key sequencial para a dimensão, começando em 1.

    row_number() numera as linhas a partir da ordenação pela chave de
    negócio, produzindo um inteiro sequencial que passa a ser a chave
    primária da dimensão na camada gold. O valor -1 fica reservado para
    o membro desconhecido (ver gerar_linha_desconhecida), por isso nunca
    é gerado aqui.

    Atenção (nota didática): como a SK é recalculada a cada execução com
    base na ordenação atual, ela pode mudar de valor entre cargas se novos
    registros forem inseridos "no meio" da ordenação. Para um exemplo de
    modelagem isso é suficiente; em um ambiente produtivo, prefira uma
    tabela de controle de SK que preserve as chaves já atribuídas.
    """
    janela = Window.orderBy(*chave_negocio)
    return df.withColumn(nome_sk, row_number().over(janela))


StatementMeta(, cf04796a-c702-461d-8ed8-9c53905e0d33, 6, Finished, Available, Finished, False)

In [5]:
def gerar_linha_desconhecida(df_gold, nome_sk: str):
    """
    Monta a linha "membro desconhecido" (SK = -1) de uma dimensão, usada
    como destino padrão quando um lançamento da fato não encontra
    correspondência (chave de negócio nula ou sem match na dimensão).

    Regra de preenchimento, aplicada genericamente a partir do schema
    da própria dimensão (não precisa listar coluna por coluna):
      - a coluna da SK recebe SK_DESCONHECIDO (-1)
      - colunas de texto (string) recebem VALOR_DESCONHECIDO_TEXTO ("N/A")
      - demais tipos (datas, booleanos, números) recebem None,
        já que "N/A" não se aplica a esses tipos
    """
    valores = {}
    for campo in df_gold.schema.fields:
        if campo.name == nome_sk:
            valores[campo.name] = SK_DESCONHECIDO
        elif isinstance(campo.dataType, StringType):
            valores[campo.name] = VALOR_DESCONHECIDO_TEXTO
        else:
            valores[campo.name] = None

    linha_desconhecida = Row(**valores)
    return spark.createDataFrame([linha_desconhecida], schema=df_gold.schema)


StatementMeta(, cf04796a-c702-461d-8ed8-9c53905e0d33, 7, Finished, Available, Finished, False)

## 3. Construir as dimensões (SK + membro desconhecido) e gravar na gold

In [6]:
# Guarda o DataFrame gold de cada dimensão (já com SK e membro desconhecido)
# para reaproveitar no join da fato, na seção 4 — evita ler a tabela gold de
# volta do disco
dims_gold = {}

for tabela_silver, cfg in CONFIG_DIMENSOES.items():
    print(f"Gerando dimensão: {cfg['tabela_gold']}")

    df_silver = spark.table(f"{LAKEHOUSE_SILVER}.{SCHEMA_SILVER}.{tabela_silver}")

    # 1. Gera a SK para os registros reais (1, 2, 3, ...)
    df_gold = gerar_dim_com_sk(df_silver, cfg["chave_negocio"], cfg["nome_sk"])

    # 2. Monta a linha do membro desconhecido (SK = -1, colunas texto = "N/A")
    #    e une com os registros reais
    df_linha_desconhecida = gerar_linha_desconhecida(df_gold, cfg["nome_sk"])
    df_gold = df_gold.unionByName(df_linha_desconhecida)

    # mergeSchema permite adicionar novas colunas à tabela Delta existente
    df_gold.write.mode("overwrite").option("mergeSchema", "true").format("delta").saveAsTable(
        f"{LAKEHOUSE_GOLD}.{SCHEMA_GOLD}.{cfg['tabela_gold']}"
    )

    dims_gold[tabela_silver] = df_gold
    print(f"  -> {df_gold.count()} linha(s) gravada(s) em {LAKEHOUSE_GOLD}.{SCHEMA_GOLD}.{cfg['tabela_gold']} (incluindo o membro desconhecido)")

print("\nDimensões geradas com sucesso.")


StatementMeta(, cf04796a-c702-461d-8ed8-9c53905e0d33, 8, Finished, Available, Finished, False)

Gerando dimensão: dim_empresa
  -> 2 linha(s) gravada(s) em lh_sap_gold.dbo.dim_empresa (incluindo o membro desconhecido)
Gerando dimensão: dim_conta_contabil
  -> 783 linha(s) gravada(s) em lh_sap_gold.dbo.dim_conta_contabil (incluindo o membro desconhecido)
Gerando dimensão: dim_segmento
  -> 4 linha(s) gravada(s) em lh_sap_gold.dbo.dim_segmento (incluindo o membro desconhecido)
Gerando dimensão: dim_centro_custo
  -> 41 linha(s) gravada(s) em lh_sap_gold.dbo.dim_centro_custo (incluindo o membro desconhecido)
Gerando dimensão: dim_centro_lucro
  -> 10 linha(s) gravada(s) em lh_sap_gold.dbo.dim_centro_lucro (incluindo o membro desconhecido)
Gerando dimensão: dim_cliente_fornecedor
  -> 117 linha(s) gravada(s) em lh_sap_gold.dbo.dim_cliente_fornecedor (incluindo o membro desconhecido)

Dimensões geradas com sucesso.


## 4. Construir a fato relacionando-se com as dimensões via SK

Para cada dimensão, trazemos só duas colunas para o join: a chave de negócio (para casar com a fato) e a SK (que é o que efetivamente queremos incorporar à fato). Depois do join, aplicamos `coalesce(sk, -1)` em cada FK, garantindo que nenhuma SK fique nula na fato — o fallback aponta para o membro desconhecido criado na seção anterior.

A fato original (ACDOCA) não tem uma coluna de data — só `ano_fiscal` e `periodo_fiscal` separados. Construímos uma coluna `data_referencia` (dia sempre `01`, ex: período 3 do ano 2024 → `2024-03-01`) a partir desses dois campos, e gravamos a tabela **particionada por ano e mês**.

In [7]:
from pyspark.sql.functions import broadcast, col, when, coalesce, lit, make_date, year, month

# Carrega a fato da camada silver
df_fato_silver = spark.table(f"{LAKEHOUSE_SILVER}.{SCHEMA_SILVER}.lancamento_despesas")

# Renomeia a coluna do parceiro de negócio para alinhar com a dimensão
if "cliente" in df_fato_silver.columns:
    df_fato_silver = df_fato_silver.withColumnRenamed("cliente", "parceiro_negocio")

# "De-para" enxuto de cada dimensão: chave de negócio + SK
# (o membro desconhecido, SK = -1, tem chave de negócio nula/"N/A" e por isso
# nunca é encontrado por um join normal — ele só entra em cena pelo coalesce
# aplicado logo abaixo)
dim_empresa_fk = (
    spark.table(f"{LAKEHOUSE_GOLD}.{SCHEMA_GOLD}.dim_empresa")
    .select("empresa", "sk_empresa")
)

dim_conta_fk = (
    spark.table(f"{LAKEHOUSE_GOLD}.{SCHEMA_GOLD}.dim_conta_contabil")
    .select("conta_contabil", "sk_conta_contabil")
)

dim_segmento_fk = (
    spark.table(f"{LAKEHOUSE_GOLD}.{SCHEMA_GOLD}.dim_segmento")
    .select("segmento", "sk_segmento")
)

dim_centro_custo_fk = (
    spark.table(f"{LAKEHOUSE_GOLD}.{SCHEMA_GOLD}.dim_centro_custo")
    .select("centro_custo", "sk_centro_custo")
)

dim_centro_lucro_fk = (
    spark.table(f"{LAKEHOUSE_GOLD}.{SCHEMA_GOLD}.dim_centro_lucro")
    .select("centro_lucro", "sk_centro_lucro")
)

# dim_cliente_fornecedor é dimensão de papéis: aqui usamos só o papel de Cliente
# (ver observação no modelo conceitual, no topo do notebook)
dim_cliente_fornecedor_fk = (
    spark.table(f"{LAKEHOUSE_GOLD}.{SCHEMA_GOLD}.dim_cliente_fornecedor")
    .select("parceiro_negocio", "sk_cliente_fornecedor")
)

# Agora construímos a fato
df_fato_gold = (
    df_fato_silver
    .join(broadcast(dim_empresa_fk), on="empresa", how="left")
    .join(broadcast(dim_conta_fk), on="conta_contabil", how="left")
    .join(broadcast(dim_segmento_fk), on="segmento", how="left")
    .join(broadcast(dim_centro_custo_fk), on="centro_custo", how="left")
    .join(broadcast(dim_centro_lucro_fk), on="centro_lucro", how="left")
    .join(broadcast(dim_cliente_fornecedor_fk), on="parceiro_negocio", how="left")
    .withColumn("sk_empresa", coalesce(col("sk_empresa"), lit(SK_DESCONHECIDO)))
    .withColumn("sk_conta_contabil", coalesce(col("sk_conta_contabil"), lit(SK_DESCONHECIDO)))
    .withColumn("sk_segmento", coalesce(col("sk_segmento"), lit(SK_DESCONHECIDO)))
    .withColumn("sk_centro_custo", coalesce(col("sk_centro_custo"), lit(SK_DESCONHECIDO)))
    .withColumn("sk_centro_lucro", coalesce(col("sk_centro_lucro"), lit(SK_DESCONHECIDO)))
    .withColumn("sk_cliente_fornecedor", coalesce(col("sk_cliente_fornecedor"), lit(SK_DESCONHECIDO)))
    .withColumn(
        "data_referencia",
        when(
            col("periodo_fiscal").cast("int").between(1, 12),
            make_date(col("ano_fiscal").cast("int"), col("periodo_fiscal").cast("int"), lit(1))
        ).otherwise(lit(None))
    )
    .withColumn("ano_particao", year(col("data_referencia")))
    .withColumn("mes_particao", month(col("data_referencia")))
    .select(
        "id_lancamento",
        "sk_empresa",
        "sk_conta_contabil",
        "sk_segmento",
        "sk_centro_custo",
        "sk_centro_lucro",
        "sk_cliente_fornecedor",
        "data_referencia",
        "periodo_fiscal",
        "ano_periodo_fiscal",
        "ano_fiscal",
        "ledger",
        "tipo_transacao",
        "valor",
        "moeda",
        "quantidade",
        "unidade_medida",
        "ano_particao",
        "mes_particao",
    )
)

# Gravação particionada por ano/mês com mergeSchema ativado: permite evoluir o schema
# da tabela fato_despesas sem erro em cargas futuras.
df_fato_gold.write.mode("overwrite") \
    .option("mergeSchema", "true") \
    .partitionBy("ano_particao", "mes_particao") \
    .format("delta") \
    .saveAsTable(f"{LAKEHOUSE_GOLD}.{SCHEMA_GOLD}.fato_despesas")

print(f"{df_fato_gold.count()} linha(s) gravada(s) em {LAKEHOUSE_GOLD}.{SCHEMA_GOLD}.fato_despesas (particionado por ano_particao, mes_particao)")


StatementMeta(, cf04796a-c702-461d-8ed8-9c53905e0d33, 9, Finished, Available, Finished, False)

504 linha(s) gravada(s) em lh_sap_gold.dbo.fato_despesas (particionado por ano_particao, mes_particao)


## 5. Criar Tabela Calendário

In [8]:
from pyspark.sql.functions import (
    col,
    lit,
    sequence,
    explode,
    year,
    month,
    dayofmonth,
    dayofweek,
    quarter,
    weekofyear,
    lpad,
    concat,
    when,
    create_map,
    current_date,
    make_date,
)
from pyspark.sql.types import StringType
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# ---------------------------------------------------------------
# 5. Criar dimensão de calendário (dim_calendario)
#   - Período: de 01/01/2026 até 31/12 do ano corrente
#   - Principais colunas: data_referencia, ano, mês, dia, nomes, trimestre, semana, etc.
# ---------------------------------------------------------------

# Datas de início e fim
ano_atual = year(current_date())
data_inicial = lit("2026-01-01").cast("date")
data_final = make_date(ano_atual, lit(12), lit(31))

# Gera sequência de datas entre data_inicial e data_final
df_datas = (
    spark.range(1)  # DataFrame base
    .select(explode(sequence(data_inicial, data_final)).alias("data_referencia"))
)

# Mapas de nomes em português (mês e dia da semana)
# dayofweek(): 1=Domingo, 2=Segunda, ..., 7=Sábado
map_dia_semana = create_map(
    lit(1), lit("Domingo"),
    lit(2), lit("Segunda-feira"),
    lit(3), lit("Terça-feira"),
    lit(4), lit("Quarta-feira"),
    lit(5), lit("Quinta-feira"),
    lit(6), lit("Sexta-feira"),
    lit(7), lit("Sábado"),
)

# Número do mês: 1 a 12
map_mes = create_map(
    lit(1), lit("Janeiro"),
    lit(2), lit("Fevereiro"),
    lit(3), lit("Março"),
    lit(4), lit("Abril"),
    lit(5), lit("Maio"),
    lit(6), lit("Junho"),
    lit(7), lit("Julho"),
    lit(8), lit("Agosto"),
    lit(9), lit("Setembro"),
    lit(10), lit("Outubro"),
    lit(11), lit("Novembro"),
    lit(12), lit("Dezembro"),
)

map_mes_abrev = create_map(
    lit(1), lit("Jan"),
    lit(2), lit("Fev"),
    lit(3), lit("Mar"),
    lit(4), lit("Abr"),
    lit(5), lit("Mai"),
    lit(6), lit("Jun"),
    lit(7), lit("Jul"),
    lit(8), lit("Ago"),
    lit(9), lit("Set"),
    lit(10), lit("Out"),
    lit(11), lit("Nov"),
    lit(12), lit("Dez"),
)

# Window para gerar SK sequencial
janela_data = Window.orderBy(col("data_referencia"))

# Criar DataFrame de calendário com colunas derivadas
df_calendario = (
    df_datas
    .withColumn("ano", year(col("data_referencia")))
    .withColumn("mes", month(col("data_referencia")))
    .withColumn("dia", dayofmonth(col("data_referencia")))
    .withColumn("numero_dia_semana", dayofweek(col("data_referencia")))
    .withColumn("nome_dia_semana", map_dia_semana[col("numero_dia_semana")])
    .withColumn("nome_mes", map_mes[col("mes")])
    .withColumn("nome_mes_abrev", map_mes_abrev[col("mes")])
    .withColumn("trimestre", quarter(col("data_referencia")))
    .withColumn("semana_ano", weekofyear(col("data_referencia")))
    .withColumn(
        "ano_mes",
        (col("ano") * lit(100) + col("mes")).cast("int")  # ex: 202603
    )
    .withColumn(
        "ano_mes_texto",
        concat(col("ano").cast(StringType()), lit("-"), lpad(col("mes").cast("string"), 2, "0"))
    )
    .withColumn(
        "eh_fim_semana",
        when(col("numero_dia_semana").isin(1, 7), lit(1)).otherwise(lit(0))
    )
)

# Criar uma SK sequencial para a data (sk_data)
df_calendario = df_calendario.withColumn("sk_data", row_number().over(janela_data))

# Reorganiza colunas (SK primeiro)
df_calendario = df_calendario.select(
    "sk_data",
    "data_referencia",
    "ano",
    "mes",
    "dia",
    "nome_mes",
    "nome_mes_abrev",
    "numero_dia_semana",
    "nome_dia_semana",
    "trimestre",
    "semana_ano",
    "ano_mes",
    "ano_mes_texto",
    "eh_fim_semana",
)

# Grava a dimensão de calendário na camada gold
df_calendario.write.mode("overwrite") \
    .option("mergeSchema", "true") \
    .format("delta") \
    .saveAsTable(f"{LAKEHOUSE_GOLD}.{SCHEMA_GOLD}.dim_calendario")

print(f"{df_calendario.count()} linha(s) gravada(s) em {LAKEHOUSE_GOLD}.{SCHEMA_GOLD}.dim_calendario")


StatementMeta(, cf04796a-c702-461d-8ed8-9c53905e0d33, 10, Finished, Available, Finished, False)

365 linha(s) gravada(s) em lh_sap_gold.dbo.dim_calendario


## 6. Checagem de qualidade — uso do membro desconhecido

Como agora nenhuma SK fica nula (o coalesce sempre resolve para -1), o sinal de atenção passa a ser a **contagem de -1** em cada coluna de FK: indica quantos lançamentos não encontraram correspondência real na dimensão (chave de negócio nula na fato, ou um código que não existe na dimensão).

In [9]:
colunas_sk = [
    "sk_empresa", "sk_conta_contabil", "sk_segmento",
    "sk_centro_custo", "sk_centro_lucro", "sk_cliente_fornecedor",
]

# Mesma técnica de agregação em uma única passada usada no notebook de silver,
# agora contando ocorrências do SK do membro desconhecido em vez de nulos
agregacoes_desconhecido = [
    spark_sum(when(col(c) == SK_DESCONHECIDO, 1).otherwise(0)).alias(c)
    for c in colunas_sk
]

relatorio_desconhecido = df_fato_gold.select(agregacoes_desconhecido).collect()[0].asDict()

print("Linhas da fato apontando para o membro desconhecido (SK = -1):")
for coluna, qtd in relatorio_desconhecido.items():
    status = "OK" if qtd == 0 else "VERIFICAR"
    print(f"  [{status}] {coluna}: {qtd}")


StatementMeta(, cf04796a-c702-461d-8ed8-9c53905e0d33, 11, Finished, Available, Finished, False)

Linhas da fato apontando para o membro desconhecido (SK = -1):
  [OK] sk_empresa: 0
  [OK] sk_conta_contabil: 0
  [OK] sk_segmento: 0
  [VERIFICAR] sk_centro_custo: 174
  [VERIFICAR] sk_centro_lucro: 24
  [VERIFICAR] sk_cliente_fornecedor: 408


## 7. Observações finais

- Todas as chaves de junção entre a fato e as dimensões, na camada gold, são feitas pela **SK** — o join usa a chave de negócio apenas como o "elo" temporário entre silver e gold; o resultado gravado na fato carrega somente as SKs, todas preenchidas (nunca nulas) graças ao membro desconhecido.
- O padrão de **membro desconhecido** (`SK = -1`, `"N/A"`) é aplicado de forma genérica a partir do schema de cada dimensão (`gerar_linha_desconhecida`), então funciona automaticamente para qualquer dimensão nova adicionada a `CONFIG_DIMENSOES` — não é preciso codificar coluna por coluna.
- `dim_cliente_fornecedor` está pronta para ser usada em mais de um papel (Cliente/Fornecedor) caso a fato passe a carregar o campo Supplier no futuro — não é necessário duplicar a tabela, só adicionar mais um join contra ela com outra coluna de SK.
- A geração de SK por `row_number()` é didática; em um pipeline incremental de produção, avalie manter uma tabela de SK persistente (ex: um `MERGE`/`upsert` que atribui SK nova só a registros de negócio inéditos, preservando o -1 como reservado).
- **`data_referencia`** é derivada de `ano_fiscal` + `periodo_fiscal` (dia sempre `01`) só para dar uma granularidade de calendário à fato, já que a extração original não trouxe uma data de lançamento. Períodos especiais de ajuste do SAP (13 a 16) não geram data válida — ficam nulos e caem na partição padrão do Delta (`__HIVE_DEFAULT_PARTITION__`). Se precisar de uma data real de lançamento, isso precisa vir da extração do ACDOCA (campo de posting date), não pode ser reconstruído aqui.
- A tabela é gravada particionada por `ano_particao`/`mes_particao` — bom para consultas que filtram por período, mas evite particionar por uma granularidade muito fina (ex: por dia) em tabelas pequenas, pois gera muitos arquivos pequenos e piora a performance em vez de melhorar.
